# **Data Science Project 1**
## Data Cleaning and Feature Engineering
"I am making this project to clean raw data and create new features so that machine learning models can run perfectly without errors."


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

print("Loading dataset...")
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

try:
    df = pd.read_csv(url)
    print("Dataset loaded successfully!")
except Exception:
    print("Network error, creating sample data.")
    np.random.seed(42)
    df = pd.DataFrame({
        "PassengerId": range(1, 101),
        "Survived": np.random.choice([0, 1], size=100),
        "Age": np.random.choice([22, 38, np.nan, 26, 35], size=100),
        "Fare": np.random.exponential(30, 100),
        "Sex": np.random.choice(["male", "female"], size=100),
        "Embarked": np.random.choice(["S", "C", np.nan], size=100)
    })

# Set target column
if "Survived" in df.columns:
    target_col = "Survived"
else:
    target_col = "SalePrice"

Loading dataset...
Dataset loaded successfully!


## Step 1: Cleaning Missing Values and Outliers
"I wrote this code to clean the data by filling missing values with median and KNN, and fixing outliers using the IQR method."

In [ ]:
# 1. Missing Values
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    missing_count = df[col].isnull().sum()
    if missing_count == 0:
        continue

    missing_pct = missing_count / len(df)

    if missing_pct < 0.05:
        df = df.dropna(subset=[col])
    elif missing_pct <= 0.20:
        df[col] = df[col].fillna(df[col].median())
    else:
        imputer = KNNImputer(n_neighbors=5)
        df[col] = imputer.fit_transform(df[[col]])

object_cols = df.select_dtypes(include="object").columns
for col in object_cols:
    if df[col].isnull().any():
        col_mode = df[col].mode()
        if len(col_mode) > 0:
            df[col] = df[col].fillna(col_mode[0])
        else:
            df[col] = df[col].fillna("Missing")

# 2. Outliers via IQR
for col in df.select_dtypes(include=np.number).columns:
    if col.lower() in ["id", "passengerid", "survived"] or df[col].nunique() <= 2:
        continue

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("Missing values and outliers cleaned!")

Missing values and outliers cleaned!


## Step 2: Encoding and Creating New Features
"I wrote this code to change text columns into numbers, delete duplicate features, and create new columns from existing data."

In [ ]:
# 1. Encoding
object_cols = df.select_dtypes(include="object").columns
for col in object_cols:
    if df[col].nunique() <= 10:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=int)
        df = pd.concat([df.drop(columns=[col]), dummies], axis=1)
    else:
        df[col] = pd.factorize(df[col])[0]

# 2. Drop High Correlation
numeric_df = df.select_dtypes(include=np.number)
if len(numeric_df.columns) >= 2:
    corr_matrix = numeric_df.corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    columns_to_drop = []
    for col in upper_tri.columns:
        if any(upper_tri[col] > 0.80):
            columns_to_drop.append(col)

    if target_col in columns_to_drop:
        columns_to_drop.remove(target_col)

    df = df.drop(columns=columns_to_drop)

# 3. Create New Features
clean_numeric = []
for c in df.select_dtypes(include=np.number).columns:
    if c.lower() not in ["id", "passengerid"] and c != target_col:
        clean_numeric.append(c)

if len(clean_numeric) >= 2:
    col1 = clean_numeric[0]
    col2 = clean_numeric[1]

    df["feat_interaction"] = df[col1] * df[col2]

    safe_denominator = df[col2].replace(0, np.nan)
    df["feat_ratio"] = df[col1] / safe_denominator
    df["feat_ratio"] = df["feat_ratio"].fillna(0)

if len(clean_numeric) >= 1:
    col1 = clean_numeric[0]
    col_mean = df[col1].mean()
    col_std = df[col1].std()
    if col_std > 0:
        df[f"{col1}_zscore"] = (df[col1] - col_mean) / col_std
    else:
        df[f"{col1}_zscore"] = 0

for col in clean_numeric:
    if df[col].nunique() > 2 and df[col].skew() > 1 and df[col].min() >= 0:
        df[f"{col}_log"] = np.log1p(df[col])
        break

# 4. Save Final Data
total_nulls = df.isnull().sum().sum()
assert total_nulls == 0, "Missing values remain!"

df.to_csv("processed_data.csv", index=False)
print("Pipeline complete! File saved as processed_data.csv")
print("Final Shape:", df.shape)

Pipeline complete! File saved as processed_data.csv
Final Shape: (891, 16)
